In [1]:
from datetime import datetime

# Obtém a data e horário atual
agora = datetime.now()

# Exibe formatado
print("Data e horário atual:", agora.strftime("%d/%m/%Y %H:%M:%S"))

Data e horário atual: 24/09/2026 11:43:59


In [2]:
"""
e3_pot_calculator_finqa_autogen.py

VERSAO 2 (apos a analise dos resultados do E2) -- mudancas que valem para
TODOS os experimentos E0..E7 (rode de novo E0 e E1 para manter a comparacao):
    - HISTORY: o historico da conversa (perguntas/respostas anteriores do
      ConvFinQA) agora vai no bloco HISTORY. Antes ia misturado no TEXT e
      HISTORY aparecia como "(none)" em 64 das 100 perguntas.
    - Leitura de JSON: pega o ULTIMO objeto JSON completo da resposta (antes
      juntava varios blocos num JSON invalido).
    - Nova metrica EM_escala_percentual: no ConvFinQA os percentuais estao
      como fracao (36.93% = 0.3693). Em perguntas de percentual/razao ela
      aceita tambem gold x100 ou gold/100. Use-a para comparar E0..E7; o EM
      estrito continua sendo reportado. O CSV ganhou as colunas question,
      is_percent_question e correct_escala_pct.
    - Programa simbolico (E2..E7): operacao "value" para consultas diretas,
      add/mul com 2 ou mais argumentos, percentuais como fracao, nomes de
      passos diferentes dos de variaveis e 4 exemplos curtos (consulta, soma,
      variacao percentual, proporcao) no lugar do exemplo unico, que era
      copiado em perguntas que nao eram de percentual. A calculadora segue a
      mesma semantica e da prioridade as variaveis quando um passo repete o
      nome de uma variavel.
    - Modo REUSE_PLANS_FROM_CSV: os planos do E2 precisam ter sido gerados
      com esta MESMA versao (semantica nova); planos da versao 1 usam a
      convencao x100 e nao sao compativeis.


EXPERIMENTO E3 - PoT + CALCULADORA DETERMINISTICA da ablacao.

Pergunta cientifica: separar planejamento de execucao ajuda? Quanto dos
erros do PoT decorre da execucao aritmetica e quanto decorre do plano?

Versao LOCAL (via Ollama). O LLM so planeja: gera o programa simbolico
(PoT). A execucao passa a ser feita por uma calculadora Python
deterministica. O LLM nao calcula mais. Nenhum critico.

Fluxo (por pergunta):

    Contexto (Question + Text + Table + History)   <- MESMA entrada do E0..E2
        |
        v
    LLM Planner   (system: mesma persona; user: MESMO prompt do E2)
        |
        v
    PoT JSON  {variables, steps[{name, op, args}], unit}
        |
        v
    Python Calculator   (recebe SO o programa, como o LLM Executor do E2)
        |
        v
    Resposta

Mantido IDENTICO ao E2 (so muda quem executa o programa):
    - AutoGen (pyautogen==0.2.35), modelo (llama3:8b), temperatura (0.2),
      CSV/amostra, MAX_ITEMS
    - system prompt (frase de persona do script de referencia)
    - montagem da entrada (build_fields: QUESTION / TEXT / TABLE / HISTORY)
    - prompt do Planner: prompt E2 do documento + semantica das operacoes
      + exemplo (flag INCLUDE_PLANNER_EXAMPLE, deve ter o MESMO valor do E2)
    - semantica das operacoes (a calculadora implementa exatamente a mesma
      definicao que foi dada ao Planner e ao LLM Executor do E2)
    - criterio de acerto (_answers_match, tol = 1e-2)

A calculadora segue a do documento de ablacao (resolve cada argumento
como variavel, resultado de passo anterior ou constante numerica e aplica
a operacao aos dois argumentos), estendida com percentage e
percentage_change, que estao na lista de operacoes permitidas do Planner.

Metricas (as mesmas do E0..E2):
    - Answer-level: EM, MAE, erro relativo medio
    - MSE / RMSE
    - Intervalo de Confianca de Wilson (95%) para o EM
    - Unit accuracy heuristica
    - Operacionais: tokens, custo hipotetico, latencia, chamadas ao LLM,
      perguntas sem resultado
    - Plano: % JSON valido, % executavel, motivos de falha, Plan Accuracy
      objetiva e decomposicao dos erros (correto | plano_invalido |
      plano_errado)

Removido em relacao ao E2 por NAO fazer sentido no E3:
    - Execution Fidelity e validade do JSON do Executor: nao ha LLM
      Executor; a calculadora e deterministica e sempre fiel ao plano.
    - Categorias erro_execucao e correto_por_acaso: sao impossiveis quando
      a resposta e a propria execucao do plano.
    - Leitura de numero em texto livre (extract_direct_answer): a resposta
      e calculada, nao lida de um texto.
    Observacao: no E3 a Plan Accuracy objetiva e igual ao EM por
    construcao (a resposta E a execucao do plano). Ela e mantida para a
    tabela da ablacao ficar com as mesmas colunas do E2.

Comparacao E2 x E3 com os MESMOS planos (opcional):
    Rodadas independentes do Planner geram planos diferentes (temperatura
    0.2), o que mistura "trocar o executor" com "sortear outro plano". Se
    REUSE_PLANS_FROM_CSV apontar para o CSV de resultados do E2, o E3 NAO
    chama o LLM: reaproveita os planos do E2 e so troca o executor. Isso
    isola exatamente a pergunta do E3. Nesse modo tokens/custo/latencia do
    Planner nao sao medidos (ficam zerados) -- use a rodada normal para
    custo.

Comparacoes entre experimentos (Kappa, McNemar) serao feitas depois, num
script separado, entre os CSVs de E0..E7 pareados por question_id, usando
a coluna 'correct' de cada um.

Saidas:
    - OUT_PATH                        -> resultados por pergunta (CSV)
    - *_metricas.csv                  -> metricas agregadas (1 linha)
    - *_per_example.jsonl             -> {"id","gold","prediction","correct",
                                          "latency","llm_calls"} por exemplo

Como rodar localmente:
    1. Instale e rode o Ollama: https://ollama.com
    2. Baixe o modelo:            ollama pull llama3:8b
    3. Confirme que o servidor esta ativo em http://localhost:11434
    4. Instale as dependencias:   pip install "pyautogen==0.2.35" pandas
       (IMPORTANTE: a partir da versao 0.4/pyautogen>=0.9 o pacote virou um
       proxy para a nova arquitetura assincrona "autogen-agentchat". Este
       script usa a API classica sincrona, por isso a versao fixada 0.2.35.)
    5. Ajuste as variaveis de configuracao abaixo se necessario
    6. Rode:                      python e3_pot_calculator_finqa_autogen.py

Requisitos:
    pip install "pyautogen==0.2.35" pandas
    (opcional) pip install scipy   -> z exato para o Intervalo de Confianca
"""

import json
import math
import re
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd

try:
    import autogen
    from autogen import AssistantAgent
except ImportError:
    print("Instale o SDK primeiro: pip install \"pyautogen==0.2.35\"", file=sys.stderr)
    raise

# scipy e usado APENAS para o valor "z" exato do Intervalo de Confianca.
# E opcional: se nao estiver instalado, o script usa valores de z
# tabelados (90/95/99%).
try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except ImportError:
    scipy_stats = None
    SCIPY_AVAILABLE = False


# ---------------------------------------------------------------------------
# CONFIGURACAO - ajuste aqui
# ---------------------------------------------------------------------------

EXPERIMENT_ID = "E3_pot_calculator"
CSV_PATH = "convfinqa_amostra_100.csv"
OLLAMA_MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"          # Ollama ignora o valor, mas o SDK exige algum
MAX_ITEMS = 100                    # quantas linhas do CSV serao lidas/processadas
TIMEOUT_SECONDS = 600              # timeout por chamada ao modelo
OUT_PATH = "results_E3_pot_calculator_autogen_v2.csv"
PER_EXAMPLE_JSONL_PATH = OUT_PATH.replace(".csv", "_per_example.jsonl")
VERBOSE = True

# Mesma temperatura do E0..E2.
LLM_TEMPERATURE = 0.2

# Inclui no prompt do Planner o exemplo do documento (receita 120 -> 150).
# DEVE ter o mesmo valor usado no E2 (e em E4..E7).
INCLUDE_PLANNER_EXAMPLE = True

# [OPCIONAL] Caminho do CSV de resultados do E2. Se preenchido, o E3 NAO
# chama o LLM: reaproveita os planos do E2 (coluna "plan") e so troca o
# executor. Isola exatamente a pergunta do E3 (mesmos planos, executor
# diferente). Deixe None para uma rodada independente (mede tokens/custo).
REUSE_PLANS_FROM_CSV: Optional[str] = None     # ex.: "results_E2_pot_symbolic_autogen_v2.csv"

# Colunas do CSV. O script de referencia usa apenas "text" (contexto +
# pergunta juntos) e "output" (gabarito). Se o seu CSV tiver colunas
# separadas para pergunta / tabela / historico (ex.: ConvFinQA), informe os
# nomes aqui para preencher cada bloco do prompt. Colunas inexistentes
# sao ignoradas automaticamente (fallback descrito em build_fields).
TEXT_COL = "text"
GOLD_COL = "output"
QUESTION_COL = "question"          # opcional
TABLE_COL = "table"                # opcional
HISTORY_COL = "history"            # opcional (ConvFinQA)

# Preco de referencia (USD por 1 milhao de tokens) usado SOMENTE para simular
# quanto custaria caso essa mesma carga de tokens fosse rodada em uma API
# paga. A execucao real deste script e local/gratuita via Ollama.
REFERENCE_API_PRICING = {
    "Groq - Llama 3.1 8B Instant": {"input_per_1m": 0.05, "output_per_1m": 0.08},
    "OpenAI - GPT-4o mini":        {"input_per_1m": 0.15, "output_per_1m": 0.60},
}


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------

# (1) System prompt: a MESMA frase de persona usada no E0..E2. Manter igual
#     em E0..E7 para que ela nao vire uma variavel da ablacao.
SYSTEM_PERSONA = """You are a financial analysis agent specializing in interpreting \
earnings reports and financial statements. Your task is to answer specific \
financial questions based on the given context from financial reports."""

# (2) Prompt do Planner: prompt E2 do documento, IDENTICO ao E2.
#     As chaves do JSON estao duplicadas ({{ }}) apenas por causa do
#     str.format().
E2_PLANNER_PROMPT = """You are a financial reasoning planner.

Do not calculate the final answer.

Generate a symbolic program capable of answering the question.

Allowed operations:

add
sub
mul
div
percentage
percentage_change

Return only JSON:

{{
  "variables": {{
  }},
  "steps": [
    {{
      "name": "",
      "op": "",
      "args": []
    }}
  ],
  "unit": ""
}}
"""

# (3) ACRESCIMO: semantica das operacoes. O documento lista as operacoes mas
#     nao define argumentos nem convencoes. VERSAO 2 (apos a analise do E2):
#     - "value" (1 argumento): perguntas de consulta direta ("qual era o
#       valor de X em 2013?") nao tinham como ser expressas -- o modelo
#       inventava operacoes ("answer", "same") ou fabricava contas;
#     - add/mul aceitam 2 ou mais argumentos (somas de varios itens);
#     - percentuais como FRACAO decimal, a convencao do ConvFinQA (o
#       gabarito de "36.93%" e 0.3693); antes multiplicava por 100;
#     - nomes de passos nao podem repetir nomes de variaveis.
#     E a MESMA semantica que a calculadora (e o LLM Executor do E2) usa.
OPERATION_SEMANTICS = """Operation semantics:
- add(a, b, ...) = a + b + ...   (two or more arguments)
- sub(a, b) = a - b
- mul(a, b, ...) = a * b * ...   (two or more arguments)
- div(a, b) = a / b
- percentage(part, whole) = part / whole
- percentage_change(old, new) = (new - old) / old
- value(a) = a   (one argument; use it when the answer is a value read directly from the context)

Rules:
- "variables" holds the numeric values taken from the context (plain numbers, no units).
- Each argument is a variable name, the name of a previous step, or a numeric constant.
- Step names must be different from variable names.
- The last step must be named "answer".
- Express percentages and ratios as decimal fractions (25% -> 0.25). Multiply by 100 only if the question explicitly asks for it.
"""

# (4) Exemplos curtos cobrindo os tipos de pergunta mais comuns (usados se
#     INCLUDE_PLANNER_EXAMPLE). VERSAO 2: o exemplo unico do documento
#     (variacao percentual) era copiado em perguntas que nao eram de
#     percentual. O exemplo do documento continua aqui (Example 3), agora
#     com percentage_change e o resultado como fracao.
PLANNER_EXAMPLE = """Examples.

Example 1 -- Question: What was the net revenue in 2019?
Relevant values: net revenue 2019 = 845.2
{{"variables": {{"net_revenue_2019": 845.2}},
  "steps": [{{"name": "answer", "op": "value", "args": ["net_revenue_2019"]}}],
  "unit": "million"}}

Example 2 -- Question: What was the combined total of the three segments?
Relevant values: segment A = 12.5, segment B = 30.1, segment C = 7.4
{{"variables": {{"segment_a": 12.5, "segment_b": 30.1, "segment_c": 7.4}},
  "steps": [{{"name": "answer", "op": "add", "args": ["segment_a", "segment_b", "segment_c"]}}],
  "unit": "million"}}

Example 3 -- Question: Revenue increased from 120 to 150. What was the percentage increase?
{{"variables": {{"old_value": 120, "new_value": 150}},
  "steps": [{{"name": "answer", "op": "percentage_change", "args": ["old_value", "new_value"]}}],
  "unit": "%"}}

Example 4 -- Question: What portion of total expenses was R&D?
Relevant values: R&D = 100, total expenses = 400
{{"variables": {{"rd": 100, "total_expenses": 400}},
  "steps": [{{"name": "answer", "op": "percentage", "args": ["rd", "total_expenses"]}}],
  "unit": "%"}}
"""

# (5) Blocos de entrada -- IDENTICOS ao E0..E2.
INPUT_BLOCKS = """QUESTION:
{question}

TEXT:
{text}

TABLE:
{table}

HISTORY:
{history}
"""

# (6) Prompt do LLM Executor (o documento so diz que, no E2, o proprio LLM
#     interpreta/executa o programa). Recebe SO o programa, como a
#     calculadora do E3.


ALLOWED_OPS = ("add", "sub", "mul", "div", "percentage", "percentage_change", "value")


# ---------------------------------------------------------------------------
# Estruturas de dados
# ---------------------------------------------------------------------------

@dataclass
class PoTCalcResult:
    question_id: object
    ground_truth: str

    # resposta final (da calculadora)
    prediction: str = ""
    correct: bool = False
    answer_source: str = ""         # calculadora | sem_resultado
    parse_error: bool = False       # True se a calculadora nao produziu resultado

    # Planner
    plan_source: str = ""           # llm | reuso_E2
    plan_json_valid: bool = False
    plan: str = ""                  # programa normalizado (JSON)
    plan_unit: str = ""
    plan_n_steps: int = 0
    planner_raw: str = ""

    # Calculadora
    calc_status: str = ""           # ok | motivo da falha
    plan_correct: bool = False      # resultado da calculadora bate com o gabarito
    error_category: str = ""        # correto | plano_invalido | plano_errado

    # tokens / custo operacional desta linha
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    latency_seconds: float = 0.0
    llm_calls: int = 0


# ---------------------------------------------------------------------------
# Utilitarios de parsing / comparacao de respostas
# ---------------------------------------------------------------------------

def _parse_json_response(text: str) -> dict:
    """Extrai o ULTIMO objeto JSON completo e valido da resposta do modelo.

    Os modelos pequenos costumam escrever texto com varios blocos JSON no
    meio (ex.: resultados parciais e depois o final). A versao anterior
    pegava tudo entre o primeiro "{" e o ultimo "}", o que juntava varios
    blocos num JSON invalido. Agora cada objeto de nivel superior e
    delimitado pelo balanceamento das chaves (ignorando chaves dentro de
    strings) e o ultimo que for valido e devolvido."""
    if text is None:
        return {"raw": "", "parse_error": True}
    candidates = []
    depth, start, in_str, esc = 0, None, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"' and depth > 0:
            in_str = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}" and depth > 0:
            depth -= 1
            if depth == 0 and start is not None:
                candidates.append(text[start:i + 1])
                start = None
    for cand in reversed(candidates):
        for attempt in (cand, cand.replace("'", '"')):
            try:
                obj = json.loads(attempt)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                return obj
    return {"raw": text, "parse_error": True}


def _extract_content(reply) -> str:
    """Extrai com seguranca o texto de uma resposta do AutoGen, que pode vir
    como str, dict ({'content': ...}) ou None (timeout/erro de conexao com
    o Ollama). Retorna sempre uma string (vazia se nao houver conteudo)."""
    if reply is None:
        return ""
    if isinstance(reply, str):
        return reply
    if isinstance(reply, dict):
        return reply.get("content", "") or ""
    return str(reply)


def _to_float(x):
    if x is None:
        return None
    s = str(x).strip().replace(",", "").replace("%", "").replace("$", "")
    try:
        return float(s)
    except ValueError:
        return None


def _answers_match(a, b, tol: float = 1e-2) -> bool:
    fa, fb = _to_float(a), _to_float(b)
    if fa is None or fb is None:
        return str(a).strip() == str(b).strip()
    return abs(fa - fb) <= tol * max(1.0, abs(fb))


# ---------------------------------------------------------------------------
# METRICAS
# ---------------------------------------------------------------------------
# Operam em cima do DataFrame de resultados (out_df) depois que todas as
# linhas foram processadas. Nao fazem nenhuma chamada ao LLM.
# Colunas usadas: prediction, correct, ground_truth.

def _pair_valid_floats(pred_series: pd.Series, gt_series: pd.Series):
    """Devolve duas listas (pred, gt) so com pares onde AMBOS os valores sao
    numericos validos. Usado para MAE / erro relativo / MSE, que nao fazem
    sentido para respostas nao-numericas ou nao-parseaveis."""
    preds, gts = [], []
    for p, g in zip(pred_series, gt_series):
        fp, fg = _to_float(p), _to_float(g)
        if fp is not None and fg is not None:
            preds.append(fp)
            gts.append(fg)
    return preds, gts


# ---------------------------------------------------------------------------
# EM com tolerancia de escala percentual
# ---------------------------------------------------------------------------
# No ConvFinQA os percentuais estao no gabarito como FRACAO (36.93% = 0.3693),
# mas alguns poucos estao em pontos percentuais. Um modelo que responde
# "36.93" para uma pergunta de percentual acerta o conteudo e erra so a
# convencao. O EM estrito continua sendo o criterio principal; o
# EM_escala_percentual aceita, SOMENTE em perguntas de percentual/razao,
# tambem prediction == gold*100 ou prediction == gold/100. E a metrica
# recomendada para comparar E0..E7, porque nao depende de cada prompt
# conhecer a convencao do dataset.

_PERCENT_Q_RE = re.compile(
    r"(?i)percent|%|\bportion\b|proportion|\bratio\b|represent|return rate|growth rate")


def is_percent_question(question: str) -> bool:
    return bool(_PERCENT_Q_RE.search(str(question or "")))


def answers_match_pct_scale(pred, gold, question: str) -> bool:
    if _answers_match(pred, gold):
        return True
    if not is_percent_question(question):
        return False
    fp, fg = _to_float(pred), _to_float(gold)
    if fp is None or fg is None:
        return False
    return _answers_match(fp, fg * 100) or _answers_match(fp, fg / 100)


def add_question_columns(out_df: pd.DataFrame, subset_df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta ao CSV a pergunta respondida, se ela e de percentual e o
    acerto com tolerancia de escala percentual."""
    if len(out_df) == 0:
        return out_df
    qs = [build_fields(subset_df.loc[qid])["question"] if qid in subset_df.index else ""
          for qid in out_df["question_id"]]
    out_df.insert(out_df.columns.get_loc("ground_truth"), "question", qs)
    out_df["is_percent_question"] = [is_percent_question(q) for q in qs]
    out_df["correct_escala_pct"] = [answers_match_pct_scale(p, g, q) for p, g, q in
                                    zip(out_df["prediction"], out_df["ground_truth"], qs)]
    return out_df


def compute_answer_level_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """Exact Match / Accuracy, MAE e erro relativo medio da resposta direta,
    comparada ao gabarito (ground_truth)."""
    metrics: Dict[str, float] = {}
    n = len(out_df)
    if n == 0:
        return metrics

    metrics["EM"] = 100.0 * out_df["correct"].sum() / n
    if "correct_escala_pct" in out_df.columns:
        metrics["EM_escala_percentual"] = 100.0 * out_df["correct_escala_pct"].sum() / n
        metrics["n_perguntas_percentual"] = int(out_df["is_percent_question"].sum())

    preds, gts = _pair_valid_floats(out_df["prediction"], out_df["ground_truth"])
    if preds:
        abs_errors = [abs(p - g) for p, g in zip(preds, gts)]
        rel_errors = [abs(p - g) / max(1.0, abs(g)) for p, g in zip(preds, gts)]
        metrics["MAE"] = sum(abs_errors) / len(abs_errors)
        metrics["ErroRelativoMedio"] = 100.0 * sum(rel_errors) / len(rel_errors)
    else:
        metrics["MAE"] = float("nan")
        metrics["ErroRelativoMedio"] = float("nan")
    metrics["n_pares_numericos"] = len(preds)
    return metrics


def compute_squared_error_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """MSE (Erro Quadratico Medio) e RMSE (sua raiz) da resposta direta.

    Complementa o MAE: o MSE eleva os erros ao quadrado antes de tirar a
    media, entao penaliza mais fortemente respostas que erram "feio" (ex.:
    o modelo erra a ordem de grandeza), enquanto o MAE trata todo desvio
    de forma linear."""
    preds, gts = _pair_valid_floats(out_df["prediction"], out_df["ground_truth"])
    if not preds:
        return {"MSE": float("nan"), "RMSE": float("nan"), "n_pares_numericos_mse": 0}
    mse = sum((p - g) ** 2 for p, g in zip(preds, gts)) / len(preds)
    return {"MSE": mse, "RMSE": math.sqrt(mse), "n_pares_numericos_mse": len(preds)}


def _z_for_confidence(confidence: float) -> float:
    """Valor critico z (distribuicao normal padrao, bicaudal) para o nivel
    de confianca pedido. Usa scipy quando disponivel (qualquer confianca);
    caso contrario, cai para uma tabela com os niveis mais comuns."""
    if SCIPY_AVAILABLE:
        return float(scipy_stats.norm.ppf(1 - (1 - confidence) / 2))
    z_table = {0.90: 1.6449, 0.95: 1.9600, 0.99: 2.5758}
    return z_table.get(round(confidence, 2), 1.9600)  # default: 95%


def _wilson_confidence_interval(successes: int, n: int, confidence: float = 0.95) -> Tuple[float, float]:
    """Intervalo de Confianca de Wilson para uma proporcao (EM/Accuracy).

    Preferido ao intervalo normal/Wald classico porque continua confiavel
    com amostras pequenas (ex.: n=100) e quando a proporcao observada esta
    perto de 0% ou 100% -- casos em que o intervalo de Wald pode extrapolar
    para fora de [0%, 100%]."""
    if n == 0:
        return float("nan"), float("nan")
    z = _z_for_confidence(confidence)
    p = successes / n
    denom = 1 + (z ** 2) / n
    centre = p + (z ** 2) / (2 * n)
    margin = z * math.sqrt((p * (1 - p) / n) + (z ** 2) / (4 * n ** 2))
    low = (centre - margin) / denom
    high = (centre + margin) / denom
    return max(0.0, low), min(1.0, high)


def compute_confidence_interval(out_df: pd.DataFrame, confidence: float = 0.95) -> Dict[str, float]:
    """Intervalo de Confianca (Wilson, 95% por padrao) para o EM.

    Com amostras pequenas (ex.: n=100), duas taxas de EM proximas (ex.:
    48% vs 45% entre E0 e E1) podem nao ser estatisticamente diferentes --
    o IC ajuda a visualizar essa incerteza."""
    n = len(out_df)
    if n == 0:
        return {}
    low, high = _wilson_confidence_interval(int(out_df["correct"].sum()), n, confidence)
    metrics = {
        "EM_IC_low": 100.0 * low,
        "EM_IC_high": 100.0 * high,
        "EM_IC_margem": 100.0 * (high - low) / 2,
        "IC_nivel_confianca": confidence,
        "IC_n_amostras": n,
    }
    if "correct_escala_pct" in out_df.columns:
        low, high = _wilson_confidence_interval(int(out_df["correct_escala_pct"].sum()), n, confidence)
        metrics["EM_escala_pct_IC_low"] = 100.0 * low
        metrics["EM_escala_pct_IC_high"] = 100.0 * high
    return metrics


def compute_unit_heuristic_accuracy(subset_df: pd.DataFrame, out_df: pd.DataFrame) -> Dict[str, float]:
    """Aproximacao HEURISTICA de Unit Accuracy (nao usa LLM): para perguntas
    cujo enunciado sugere uma unidade (%, milhao, bilhao), verifica se a
    resposta esta na mesma ordem de grandeza do gabarito (desvio relativo
    < 50%). Um erro de unidade (ex.: 0.25 em vez de 25%) cai fora dessa
    faixa. E um proxy grosseiro.

    As linhas sao pareadas por question_id (indice do CSV), entao linhas
    que falharam durante a execucao nao desalinham a comparacao."""
    def unit_hint(text: str) -> Optional[str]:
        t = str(text).lower()
        if "%" in t or "percent" in t:
            return "percent"
        if "billion" in t:
            return "billion"
        if "million" in t:
            return "million"
        return None

    matches = 0
    total = 0
    for _, row in out_df.iterrows():
        qid = row["question_id"]
        if qid not in subset_df.index:
            continue
        if unit_hint(subset_df.at[qid, TEXT_COL]) is None:
            continue
        total += 1
        fa, fg = _to_float(row["prediction"]), _to_float(row["ground_truth"])
        if fa is None or fg is None:
            continue
        if fg != 0 and abs(fa - fg) / max(1.0, abs(fg)) < 0.5:
            matches += 1
    return {
        "unit_accuracy_heuristica": 100.0 * matches / total if total else float("nan"),
        "n_perguntas_com_unidade_detectada": total,
    }


def compute_plan_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """Metricas do plano (mesmos nomes do E2, exceto as do LLM Executor):

    - plano_json_valido_pct / plano_executavel_pct: conformidade do Planner.
    - plan_accuracy_objetiva: % de todas as perguntas cujo plano, executado
      pela calculadora, bate com o gabarito (= EM no E3, por construcao).
    - distribuicao de error_category (em % de todas as perguntas).
    - contagem dos motivos de plano nao executavel.
    """
    n = len(out_df)
    if n == 0:
        return {}
    executable = out_df["calc_status"] == "ok"
    metrics: Dict[str, float] = {
        "plano_json_valido_pct": 100.0 * out_df["plan_json_valid"].sum() / n,
        "plano_executavel_pct": 100.0 * executable.sum() / n,
        "plan_accuracy_objetiva": 100.0 * out_df["plan_correct"].sum() / n,
    }
    for cat in ("correto", "plano_invalido", "plano_errado"):
        metrics[f"cat_{cat}_pct"] = 100.0 * (out_df["error_category"] == cat).sum() / n
    for status, cnt in out_df.loc[~executable, "calc_status"].value_counts().items():
        metrics[f"plano_falha_{status}"] = int(cnt)
    return metrics


# ---------------------------------------------------------------------------
# Configuracao AutoGen
# ---------------------------------------------------------------------------

def _base_llm_config(temperature: float, model: str = OLLAMA_MODEL) -> dict:
    return {
        "config_list": [{
            "model": model,
            "base_url": OLLAMA_BASE_URL,
            "api_key": OLLAMA_API_KEY,
        }],
        "temperature": temperature,
        "timeout": TIMEOUT_SECONDS,
        "cache_seed": None,  # desliga cache do autogen p/ nao mascarar tokens/custos
    }



def _usage_totals(agents: List) -> Dict[str, int]:
    """Soma prompt/completion/total tokens de uma lista de agentes autogen,
    usando autogen.gather_usage_summary (cobre todos os modelos usados)."""
    summary = autogen.gather_usage_summary(agents)
    totals = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    usage = summary.get("usage_including_cached_inference", {})
    for model_name, stats in usage.items():
        if model_name == "total_cost":
            continue
        totals["prompt_tokens"] += stats.get("prompt_tokens", 0)
        totals["completion_tokens"] += stats.get("completion_tokens", 0)
        totals["total_tokens"] += stats.get("total_tokens", 0)
    return totals




# ---------------------------------------------------------------------------
# Montagem da entrada (QUESTION / TEXT / TABLE / HISTORY) -- identica ao E0/E1
# ---------------------------------------------------------------------------

_FIRST_TURN_RE = re.compile(r"(?i)(?:^|\n)[ \t]*question[ \t]*:")
_TURN_RE = re.compile(
    r"(?is)question[ \t]*:[ \t]*(.*?)(?:\n[ \t]*answer[ \t]*:[ \t]*(.*?))?[ \t]*(?=\n[ \t]*question[ \t]*:|\Z)")


def _cell(row: pd.Series, col: Optional[str]) -> Optional[str]:
    """Le uma coluna opcional do CSV; devolve None se nao existir/vazia."""
    if not col or col not in row.index:
        return None
    val = row[col]
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    s = str(val).strip()
    return s or None


def split_dialogue(full_text: str) -> Tuple[str, List[Tuple[str, str]], Optional[str]]:
    """Separa o texto do ConvFinQA em (contexto, historico, ultima pergunta).

    No CSV, a coluna "text" traz o relatorio (texto + tabela) seguido do
    dialogo: "Question: ... Answer: ... Question: ...". A ultima pergunta
    (sem resposta) e a que deve ser respondida; os pares anteriores sao o
    historico da conversa."""
    m = _FIRST_TURN_RE.search(full_text)
    if not m:
        return full_text.strip(), [], None
    context = full_text[:m.start()].strip()
    turns = [(q.strip(), (a or "").strip()) for q, a in _TURN_RE.findall(full_text[m.start():])]
    turns = [t for t in turns if t[0]]
    if not turns:
        return context, [], None
    return context, turns[:-1], turns[-1][0]


def build_fields(row: pd.Series) -> Dict[str, str]:
    """Preenche os blocos QUESTION / TEXT / TABLE / HISTORY (igual em E0..E7).

    - Colunas separadas (QUESTION_COL, TABLE_COL, HISTORY_COL), se existirem
      no CSV, sao usadas diretamente.
    - Caso contrario, a coluna "text" e dividida por split_dialogue():
      TEXT = relatorio (texto + tabela), QUESTION = ultima pergunta do
      dialogo e HISTORY = perguntas e respostas anteriores. Na versao
      anterior o historico ia misturado no TEXT e HISTORY aparecia como
      "(none)" mesmo em conversas de varios turnos.
    - Se nao houver "Question:" no texto, tudo vai para TEXT e QUESTION
      indica que a pergunta esta nele. Nenhuma informacao e descartada.
    """
    full_text = _cell(row, TEXT_COL) or ""
    question = _cell(row, QUESTION_COL)
    table = _cell(row, TABLE_COL)
    history = _cell(row, HISTORY_COL)

    text = full_text
    if question is None:
        context, turns, last_q = split_dialogue(full_text)
        if last_q:
            text, question = context, last_q
            if history is None and turns:
                history = "\n".join(f"Question: {q}\nAnswer: {a}" for q, a in turns)
        else:
            question = "(The question is included at the end of the TEXT block.)"

    return {
        "question": question,
        "text": text if text else "(not provided)",
        "table": table if table else "(not provided separately -- any table is included in the TEXT block)",
        "history": history if history else "(none)",
    }


def _as_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    return str(x)


# ---------------------------------------------------------------------------
# Programa simbolico: normalizacao e CALCULADORA deterministica
# ---------------------------------------------------------------------------
# Mesmo codigo que no E2 servia apenas de executor de referencia (medicao).
# No E3 ele passa a produzir a resposta. Segue a calculadora do documento
# de ablacao: resolve cada argumento (variavel -> resultado de passo
# anterior -> constante numerica) e aplica a operacao aos dois argumentos.

def normalize_plan(parsed: dict) -> dict:
    """Mantem apenas variables / steps / unit do JSON do Planner."""
    variables = parsed.get("variables")
    steps = parsed.get("steps")
    return {
        "variables": variables if isinstance(variables, dict) else {},
        "steps": steps if isinstance(steps, list) else [],
        "unit": _as_text(parsed.get("unit")),
    }


def _apply_op(op: str, vals: List[float]) -> float:
    if op == "value":
        return vals[0]
    if op == "add":
        return sum(vals)
    if op == "mul":
        out = 1.0
        for v in vals:
            out *= v
        return out
    a, b = vals
    if op == "sub":
        return a - b
    if op == "div":
        return a / b
    if op == "percentage":
        return a / b
    if op == "percentage_change":
        return (b - a) / a
    raise ValueError(op)


_ARITY = {"value": (1, 1), "add": (2, None), "mul": (2, None), "sub": (2, 2),
          "div": (2, 2), "percentage": (2, 2), "percentage_change": (2, 2)}


def execute_program(plan: dict) -> Tuple[Optional[float], str]:
    """Calculadora: executa o programa deterministicamente. Devolve
    (resultado, status).

    status: ok | sem_steps | variavel_nao_numerica | step_malformado |
            op_invalida | aridade_invalida | arg_nao_resolvido |
            divisao_por_zero | erro_numerico
    O resultado e o do step chamado "answer" ou, se nao houver, o do ultimo.

    Resolucao de argumentos: variavel -> resultado de passo anterior ->
    constante numerica. As VARIAVEIS tem prioridade: se um passo reutiliza
    o nome de uma variavel (proibido no prompt, mas acontece), o nome
    continua apontando para o valor extraido do contexto.
    """
    steps = plan.get("steps") or []
    if not steps:
        return None, "sem_steps"

    variables: Dict[str, float] = {}
    for k, v in (plan.get("variables") or {}).items():
        fv = _to_float(v)
        if fv is None:
            return None, "variavel_nao_numerica"
        variables[str(k)] = fv

    results: Dict[str, float] = {}
    last = None
    for step in steps:
        if not isinstance(step, dict):
            return None, "step_malformado"
        op = str(step.get("op", "")).strip().lower()
        args = step.get("args")
        name = str(step.get("name", "")).strip() or f"step_{len(results)}"
        if op not in ALLOWED_OPS:
            return None, "op_invalida"
        lo, hi = _ARITY[op]
        if not isinstance(args, list) or len(args) < lo or (hi is not None and len(args) > hi):
            return None, "aridade_invalida"

        values = []
        for arg in args:
            key = str(arg).strip()
            if key in variables:
                values.append(variables[key])
            elif key in results:
                values.append(results[key])
            elif _to_float(arg) is not None:
                values.append(_to_float(arg))
            else:
                return None, "arg_nao_resolvido"
        try:
            last = _apply_op(op, values)
        except ZeroDivisionError:
            return None, "divisao_por_zero"
        except (ValueError, OverflowError):
            return None, "erro_numerico"
        results[name] = last

    return results.get("answer", last), "ok"


# ---------------------------------------------------------------------------
# Prompts preenchidos
# ---------------------------------------------------------------------------

def build_planner_prompt(row: pd.Series) -> str:
    template = E2_PLANNER_PROMPT + "\n" + OPERATION_SEMANTICS + "\n"
    if INCLUDE_PLANNER_EXAMPLE:
        template += PLANNER_EXAMPLE + "\n"
    template += INPUT_BLOCKS
    return template.format(**build_fields(row))


# ---------------------------------------------------------------------------
# Configuracao do agente
# ---------------------------------------------------------------------------

def build_agents():
    planner = AssistantAgent(
        name="PoT_Planner",
        system_message=SYSTEM_PERSONA,
        llm_config=_base_llm_config(LLM_TEMPERATURE),
        human_input_mode="NEVER",
        max_consecutive_auto_reply=1,
        code_execution_config=False,
    )
    return planner


def load_reused_plans(path: str) -> Dict[str, dict]:
    """Le os planos do CSV de resultados do E2, indexados por question_id."""
    df = pd.read_csv(path)
    plans: Dict[str, dict] = {}
    for _, r in df.iterrows():
        plans[str(r["question_id"])] = {
            "plan_json_valid": bool(r["plan_json_valid"]),
            "plan": r["plan"] if isinstance(r["plan"], str) else "",
            "planner_raw": r["planner_raw"] if isinstance(r.get("planner_raw"), str) else "",
        }
    return plans


# ---------------------------------------------------------------------------
# Fluxo E3: Contexto -> LLM Planner -> PoT JSON -> Calculadora -> Resposta
# ---------------------------------------------------------------------------

def _categorize(correct: bool, executable: bool) -> str:
    if correct:
        return "correto"
    return "plano_errado" if executable else "plano_invalido"


def run_pot_calc(planner, row: pd.Series, ground_truth: str,
                 reused: Optional[dict] = None,
                 verbose: bool = False) -> PoTCalcResult:
    result = PoTCalcResult(question_id=None, ground_truth=str(ground_truth))
    t0 = time.perf_counter()

    # --- 1) Planner gera o programa simbolico (ou reaproveita o do E2) ---
    if reused is not None:
        result.plan_source = "reuso_E2"
        result.planner_raw = reused["planner_raw"]
        parsed = json.loads(reused["plan"]) if reused["plan_json_valid"] and reused["plan"] else {"parse_error": True}
    else:
        result.plan_source = "llm"
        reply = planner.generate_reply(messages=[{"role": "user", "content": build_planner_prompt(row)}])
        result.llm_calls += 1
        result.planner_raw = _extract_content(reply)
        parsed = _parse_json_response(result.planner_raw)
    result.plan_json_valid = not parsed.get("parse_error", False)

    # --- 2) Calculadora executa o programa ---
    answer = None
    if result.plan_json_valid:
        plan = normalize_plan(parsed)
        result.plan = json.dumps(plan, ensure_ascii=False)
        result.plan_unit = plan["unit"]
        result.plan_n_steps = len(plan["steps"])
        answer, result.calc_status = execute_program(plan)
    else:
        result.calc_status = "plano_json_invalido"

    result.latency_seconds = time.perf_counter() - t0
    result.prediction = str(answer)
    result.answer_source = "calculadora" if answer is not None else "sem_resultado"
    result.parse_error = answer is None
    result.correct = _answers_match(result.prediction, ground_truth)
    result.plan_correct = result.correct
    result.error_category = _categorize(result.correct, result.calc_status == "ok")

    if verbose:
        print(f"    [PoT_Planner] ({result.plan_source}) json_valido={result.plan_json_valid} "
              f"steps={result.plan_n_steps}")
        print(f"    [Calculator] -> {result.prediction} ({result.calc_status}) | "
              f"categoria: {result.error_category}")
    return result


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def _fmt(x, spec=".2f"):
    try:
        return "N/D" if x is None or (isinstance(x, float) and math.isnan(x)) else format(x, spec)
    except (TypeError, ValueError):
        return str(x)


def main():
    df = pd.read_csv(CSV_PATH)
    subset = df.head(MAX_ITEMS)

    results: List[PoTCalcResult] = []
    n_correct = 0

    print(f"Experimento: {EXPERIMENT_ID}")
    print(f"Modelo: {OLLAMA_MODEL} | Endpoint: {OLLAMA_BASE_URL}")
    print(f"Processando {len(subset)} de {len(df)} linhas do CSV (MAX_ITEMS={MAX_ITEMS})")
    reused_plans = load_reused_plans(REUSE_PLANS_FROM_CSV) if REUSE_PLANS_FROM_CSV else None
    if reused_plans is not None:
        print(f"MODO REUSO: planos lidos de {REUSE_PLANS_FROM_CSV} "
              f"({len(reused_plans)} planos). Nenhuma chamada ao LLM; tokens/custo nao medidos.")
    print(f"PoT simbolico: LLM Planner (temperatura {LLM_TEMPERATURE}) + calculadora Python | "
          f"1 chamada por pergunta | exemplo no prompt: {INCLUDE_PLANNER_EXAMPLE} | "
          "sem critico\n")

    for i, (idx, row) in enumerate(subset.iterrows(), start=1):
        ground_truth = row[GOLD_COL]

        print(f"[{i}/{len(subset)}] Processando linha id={idx} ...")

        planner = build_agents()
        agents = [planner]
        usage_before = _usage_totals(agents)

        reused = None
        if reused_plans is not None:
            reused = reused_plans.get(str(idx))
            if reused is None:
                print(f"  AVISO: question_id={idx} nao encontrado no CSV do E2; pulando.",
                      file=sys.stderr)
                continue

        try:
            result = run_pot_calc(planner, row, ground_truth, reused=reused, verbose=VERBOSE)
        except Exception as e:
            print(f"  ERRO: {e}", file=sys.stderr)
            continue

        usage_after = _usage_totals(agents)
        result.question_id = idx
        result.prompt_tokens = usage_after["prompt_tokens"] - usage_before["prompt_tokens"]
        result.completion_tokens = usage_after["completion_tokens"] - usage_before["completion_tokens"]
        result.total_tokens = usage_after["total_tokens"] - usage_before["total_tokens"]

        results.append(result)
        n_correct += int(result.correct)

        print(f"  -> prediction: {result.prediction} (correto: {result.correct}) | "
              f"gt: {result.ground_truth} | tokens: {result.total_tokens} | "
              f"latencia: {result.latency_seconds:.2f}s\n")

        time.sleep(0.1)

    # ------------------------------------------------------------------
    # Salva resultados detalhados (1 linha por pergunta)
    # ------------------------------------------------------------------
    out_df = pd.DataFrame([{
        "experiment": EXPERIMENT_ID,
        "question_id": r.question_id,
        "ground_truth": r.ground_truth,
        "prediction": r.prediction,
        "correct": r.correct,
        "answer_source": r.answer_source,
        "parse_error": r.parse_error,
        "error_category": r.error_category,
        "plan_source": r.plan_source,
        "plan_json_valid": r.plan_json_valid,
        "plan": r.plan,
        "plan_unit": r.plan_unit,
        "plan_n_steps": r.plan_n_steps,
        "calc_status": r.calc_status,
        "plan_correct": r.plan_correct,
        "planner_raw": r.planner_raw,
        "prompt_tokens": r.prompt_tokens,
        "completion_tokens": r.completion_tokens,
        "total_tokens": r.total_tokens,
        "latency_seconds": r.latency_seconds,
        "llm_calls": r.llm_calls,
    } for r in results])
    out_df = add_question_columns(out_df, subset)
    out_df.to_csv(OUT_PATH, index=False)

    # Registro por exemplo no formato pedido no documento de ablacao.
    with open(PER_EXAMPLE_JSONL_PATH, "w", encoding="utf-8") as f:
        for r in results:
            gold_f, pred_f = _to_float(r.ground_truth), _to_float(r.prediction)
            f.write(json.dumps({
                "id": str(r.question_id),
                "gold": gold_f if gold_f is not None else r.ground_truth,
                "prediction": pred_f if pred_f is not None else (None if r.prediction == "None" else r.prediction),
                "correct": bool(r.correct),
                "latency": round(r.latency_seconds, 3),
                "llm_calls": r.llm_calls,
            }, ensure_ascii=False) + "\n")

    n = len(results)

    print("=" * 70)
    print(f"RESUMO - {EXPERIMENT_ID}: PoT simbolico + calculadora (AutoGen)")
    print("=" * 70)
    if n == 0:
        print("Nenhuma linha processada com sucesso.")
        return

    total_prompt_tokens = sum(r.prompt_tokens for r in results)
    total_completion_tokens = sum(r.completion_tokens for r in results)
    total_tokens = sum(r.total_tokens for r in results)
    n_parse_errors = sum(int(r.parse_error) for r in results)
    total_latency = sum(r.latency_seconds for r in results)
    total_calls = sum(r.llm_calls for r in results)

    print(f"Linhas avaliadas:                          {n}")
    print(f"Perguntas sem resultado da calculadora:    {n_parse_errors}/{n} "
          f"({100 * n_parse_errors / n:.2f}%)")

    # ------------------------------------------------------------------
    # Metricas
    # ------------------------------------------------------------------
    answer_metrics = compute_answer_level_metrics(out_df)
    squared_error_metrics = compute_squared_error_metrics(out_df)
    ci_metrics = compute_confidence_interval(out_df, confidence=0.95)
    unit_metrics = compute_unit_heuristic_accuracy(subset, out_df)
    plan_metrics = compute_plan_metrics(out_df)

    print("\n" + "-" * 70)
    print("ACERTOS")
    print("-" * 70)
    print(f"Acertos: {n_correct}/{n} -> EM = {_fmt(answer_metrics.get('EM'))}%")
    print(f"IC {int(ci_metrics.get('IC_nivel_confianca', 0.95) * 100)}% (Wilson): "
          f"[{_fmt(ci_metrics.get('EM_IC_low'))}%, {_fmt(ci_metrics.get('EM_IC_high'))}%] "
          f"(margem +-{_fmt(ci_metrics.get('EM_IC_margem'))} p.p.)")
    print(f"EM com escala percentual: {_fmt(answer_metrics.get('EM_escala_percentual'))}% "
          f"[{_fmt(ci_metrics.get('EM_escala_pct_IC_low'))}%, {_fmt(ci_metrics.get('EM_escala_pct_IC_high'))}%] "
          f"(aceita gold x100 ou /100 nas {answer_metrics.get('n_perguntas_percentual', 0)} perguntas "
          "de percentual; use esta para comparar E0..E7)")

    print("\n" + "-" * 70)
    print("ERRO NUMERICO (apenas pares com resposta numerica valida)")
    print("-" * 70)
    print(f"MAE:                 {_fmt(answer_metrics.get('MAE'), '.4f')} "
          f"(n={answer_metrics.get('n_pares_numericos', 0)})")
    print(f"Erro relativo medio: {_fmt(answer_metrics.get('ErroRelativoMedio'))}%")
    print(f"MSE:                 {_fmt(squared_error_metrics.get('MSE'), '.4f')}")
    print(f"RMSE:                {_fmt(squared_error_metrics.get('RMSE'), '.4f')}")

    print("\n" + "-" * 70)
    print("UNIDADE (heuristico, sem LLM)")
    print("-" * 70)
    print(f"Unit accuracy heuristica: {_fmt(unit_metrics.get('unit_accuracy_heuristica'))}% "
          f"(n={unit_metrics.get('n_perguntas_com_unidade_detectada', 0)})")

    print("\n" + "-" * 70)
    print("PLANO SIMBOLICO")
    print("-" * 70)
    print(f"Plano em JSON valido:         {_fmt(plan_metrics.get('plano_json_valido_pct'))}%")
    print(f"Plano executavel:             {_fmt(plan_metrics.get('plano_executavel_pct'))}%")
    print(f"Plan Accuracy (objetiva):     {_fmt(plan_metrics.get('plan_accuracy_objetiva'))}%  "
          "(= EM no E3: a resposta e a execucao do plano)")
    falhas = {k: v for k, v in plan_metrics.items() if k.startswith("plano_falha_")}
    if falhas:
        print("Motivos de plano nao executavel:")
        for k, v in sorted(falhas.items(), key=lambda kv: -kv[1]):
            print(f"    {k.replace('plano_falha_', ''):<24} {v}")

    print("\n" + "-" * 70)
    print("DECOMPOSICAO DOS RESULTADOS (% de todas as perguntas)")
    print("-" * 70)
    print(f"correto:                      {_fmt(plan_metrics.get('cat_correto_pct'))}%")
    print(f"plano_invalido:               {_fmt(plan_metrics.get('cat_plano_invalido_pct'))}%")
    print(f"plano_errado:                 {_fmt(plan_metrics.get('cat_plano_errado_pct'))}%")
    print("(erro de execucao e impossivel no E3: a calculadora e deterministica)")

    print("\n" + "-" * 70)
    print("CUSTO OPERACIONAL")
    print("-" * 70)
    print(f"Chamadas ao LLM:              {total_calls} ({total_calls / n:.2f} por pergunta)")
    print(f"Latencia total:               {total_latency:,.1f}s")
    print(f"Latencia media por pergunta:  {total_latency / n:,.2f}s")
    print(f"Prompt tokens (entrada):      {total_prompt_tokens:,}")
    print(f"Completion tokens (saida):    {total_completion_tokens:,}")
    print(f"Total tokens:                 {total_tokens:,}")
    print(f"Media de tokens por pergunta: {total_tokens / n:,.1f}")

    print("\n" + "-" * 70)
    print("COMPARACAO DE CUSTO HIPOTETICO (execucao real foi LOCAL/GRATUITA via Ollama)")
    print("-" * 70)
    print("Os valores abaixo simulam quanto custaria rodar o MESMO volume de "
          "tokens em uma API paga (precos de referencia, sujeitos a mudanca):\n")
    cost_metrics: Dict[str, float] = {}
    for provider, price in REFERENCE_API_PRICING.items():
        cost_input = (total_prompt_tokens / 1_000_000) * price["input_per_1m"]
        cost_output = (total_completion_tokens / 1_000_000) * price["output_per_1m"]
        cost_total = cost_input + cost_output
        cost_metrics[f"custo_hipotetico_usd__{provider}"] = cost_total
        print(f"  {provider}:")
        print(f"    ${price['input_per_1m']:.3f}/1M input, ${price['output_per_1m']:.3f}/1M output")
        print(f"    Custo estimado: ${cost_input:.4f} (input) + ${cost_output:.4f} (output) "
              f"= ${cost_total:.4f} total | ${cost_total / n:.5f} por pergunta")
    print("\n  Ollama (local, GPU/CPU proprios): $0.00 por chamada de API "
          "(custo real e apenas energia/hardware).")

    operational_metrics = {
        "experiment": EXPERIMENT_ID,
        "modelo": OLLAMA_MODEL,
        "temperatura": LLM_TEMPERATURE,
        "modo_reuso_planos_E2": bool(REUSE_PLANS_FROM_CSV),
        "n_avaliadas": n,
        "n_sem_numero_extraivel": n_parse_errors,
        "llm_calls_total": total_calls,
        "llm_calls_por_pergunta": total_calls / n,
        "latencia_total_s": total_latency,
        "latencia_media_s": total_latency / n,
        "prompt_tokens_total": total_prompt_tokens,
        "completion_tokens_total": total_completion_tokens,
        "total_tokens": total_tokens,
        "tokens_medios_por_pergunta": total_tokens / n,
    }

    metrics_path = OUT_PATH.replace(".csv", "_metricas.csv")
    all_metrics = {
        **operational_metrics,
        **answer_metrics,
        **squared_error_metrics,
        **ci_metrics,
        **unit_metrics,
        **plan_metrics,
        **cost_metrics,
    }

    pd.DataFrame([all_metrics]).to_csv(metrics_path, index=False)

    print(f"\nResultados detalhados salvos em: {OUT_PATH}")
    print(f"Metricas agregadas salvas em:    {metrics_path}")
    print(f"Registro por exemplo salvo em:   {PER_EXAMPLE_JSONL_PATH}")


if __name__ == "__main__":
    main()


Experimento: E3_pot_calculator
Modelo: llama3:8b | Endpoint: http://localhost:11434/v1
Processando 100 de 100 linhas do CSV (MAX_ITEMS=100)
PoT simbolico: LLM Planner (temperatura 0.2) + calculadora Python | 1 chamada por pergunta | exemplo no prompt: True | sem critico

[1/100] Processando linha id=0 ...
[autogen.oai.client: 09-24 11:44:18] {744} WARNING - Model llama3:8b is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
    [PoT_Planner] (llm) json_valido=True steps=3
    [Calculator] -> 0.0 (ok) | categoria: plano_errado
  -> prediction: 0.0 (correto: False) | gt: 93.4 | tokens: 1943 | latencia: 12.75s

[2/100] Processando linha id=1 ...
[autogen.oai.client: 09-24 11:44:24] {744} WARNING - Model llama3:8b is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
    [PoT_Planner] (llm) 

In [3]:
from datetime import datetime

# Obtém a data e horário atual
agora = datetime.now()

# Exibe formatado
print("Data e horário atual:", agora.strftime("%d/%m/%Y %H:%M:%S"))

Data e horário atual: 24/09/2026 11:57:04
